In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login

notebook_login()
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [2]:
ds = load_dataset("openai/gsm8k", "main")
train_ds = ds["train"]
test_ds = ds["test"]

In [3]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [4]:
def tokenize(element, tokenizer, is_eval=False):
    prompt = "Answer the math question step-by-step. End your response by clearly stating your final answer."
    qs = element["question"]
    ans = element.get("answer", None)
    prompt_message = [
        [
            {"role": "system", "content": prompt},
            {"role": "user", "content": q},
        ]
        for q in qs
    ]
    tokens_prompt = tokenizer.apply_chat_template(prompt_message, tokenize=True, add_generation_prompt=True, return_dict=True)
    if is_eval:
        targets = [[-100]*len(ids) for ids in tokens_prompt["input_ids"]]
        return {"input_ids": tokens_prompt["input_ids"], "labels": targets, "attention_mask": tokens_prompt["attention_mask"]}
    else:
        train_message = [
            prompt_message[i] + [{"role": "assistant", "content": a}]
            for i, a in enumerate(ans)
        ]
        tokens_train = tokenizer.apply_chat_template(train_message, tokenize=True, add_generation_prompt=False, return_dict=True)
        targets = [
            ([-100] * len(p)) + t[len(p):]
            for p, t in zip(tokens_prompt["input_ids"], tokens_train["input_ids"])
        ]
        return {"input_ids": tokens_train["input_ids"], "labels": targets, "attention_mask": tokens_train["attention_mask"]}

In [5]:
def tokenize_2(element, tokenizer, is_eval=False):
    prompt = "Answer the math question step-by-step. End your response by clearly stating your final answer."
    qs = element["question"]
    ans = element.get("answer", None)
    prompt_message = [
        [
            {"role": "system", "content": prompt},
            {"role": "user", "content": q},
        ]
        for q in qs
    ]
    tokens_prompt = tokenizer.apply_chat_template(prompt_message, tokenize=True, add_generation_prompt=True, return_dict=True)
    train_message = [
            prompt_message[i] + [{"role": "assistant", "content": a}]
            for i, a in enumerate(ans)
        ]
    tokens_train = tokenizer.apply_chat_template(train_message, tokenize=True, add_generation_prompt=False, return_dict=True)
    targets = [
            ([-100] * len(p)) + t[len(p):]
            for p, t in zip(tokens_prompt["input_ids"], tokens_train["input_ids"])
        ]
    if is_eval:
        return {"input_ids": tokens_prompt["input_ids"], "labels": targets, "attention_mask": tokens_prompt["attention_mask"]}
    else:
        return {"input_ids": tokens_train["input_ids"], "labels": targets, "attention_mask": tokens_train["attention_mask"]}

In [6]:
tokenized_train = ds['train'].map(
    tokenize,
    batched=True,
    remove_columns=ds["train"].column_names,
    fn_kwargs={"tokenizer":tokenizer, "is_eval":False},
    load_from_cache_file=False
)
tokenized_test = ds['test'].map(
    tokenize,
    batched=True,
    remove_columns=ds["test"].column_names,
    fn_kwargs={"tokenizer":tokenizer, "is_eval":True},
    load_from_cache_file=False
)

Map: 100%|##########| 7473/7473 [00:00<?, ? examples/s]

Map: 100%|##########| 1319/1319 [00:00<?, ? examples/s]

In [7]:
# Code for the collator: enables batching
def get_data_collator(tokenizer):
    def data_collator(features):
        is_eval = "labels" not in features[0]
        feats = [{k: v for k, v in f.items() if k != "labels"} for f in features]
        batch = tokenizer.pad(
            feats,
            padding=True,
            return_tensors="pt",
        )
        if not is_eval:
            max_len = batch["input_ids"].shape[1]
            labels = [f["labels"] for f in features]
            padded_labels = []
            for lab in labels:
                pad_len = max_len - len(lab)
                if tokenizer.padding_side == "left":
                    padded_lab = ([-100] * pad_len) + lab
                else:
                    padded_lab = lab + ([-100] * pad_len)
                padded_labels.append(padded_lab)
            batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch
    return data_collator

In [8]:
config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)
lora_model = get_peft_model(model, config)

In [9]:
# args = Seq2SeqTrainingArguments(
#     output_dir='notebook-demo',
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     logging_strategy="epoch",
#     gradient_accumulation_steps=8,
#     num_train_epochs=1,
#     weight_decay=0.1,
#     warmup_steps=1_000,
#     lr_scheduler_type="cosine",
#     learning_rate=5e-4,
# )

fast_eval_args = Seq2SeqTrainingArguments(
    output_dir='notebook-demo',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    weight_decay=0.1,
    warmup_steps=1_000,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
)

In [10]:
trainer = Seq2SeqTrainer(
    model=lora_model,
    processing_class=tokenizer,
    args=fast_eval_args,
    data_collator=get_data_collator(tokenizer),
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [29]:
def demo(compute_loss_function):
    args = Seq2SeqTrainingArguments(
        output_dir='tmp',
        per_device_train_batch_size=1,
        save_strategy="epoch",
        logging_strategy="epoch",
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        weight_decay=0.1,
        warmup_steps=1_000,
        lr_scheduler_type="cosine",
        learning_rate=5e-4,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        processing_class=tokenizer,          # or processing_class in newer versions
        data_collator=get_data_collator(tokenizer),
        train_dataset=tokenized_train.select(range(3)),
        compute_loss_func=None,
    )

    out = trainer.train() 
    return out

In [35]:
import torch.nn.functional as F

def compute_loss_function(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits[..., :-1, :].contiguous()
    labels = labels[..., 1:].contiguous()
    log_probs = -F.log_softmax(logits, dim=-1)
    labels = labels.unsqueeze(-1)

    padding_mask = labels.eq(-100)
    # In case the ignore_index is -100, the gather will fail, so we replace labels by 0. The padding_mask
    # will ignore them in any case.
    labels = torch.clamp(labels, min=0)
    nll_loss = log_probs.gather(dim=-1, index=labels)

    nll_loss.masked_fill_(padding_mask, 0.0)

    # Take the mean over the label dimensions, then divide by the number of active elements (i.e. not-padded):
    num_active_elements = padding_mask.numel() - padding_mask.long().sum()
    nll_loss = nll_loss.sum() / num_active_elements
    return  nll_loss

def compute_loss_function(outputs, labels, num_items_in_batch=None):
    # Shift the logits and labels such that they are aligned for next-token comparison
    logits = outputs.logits[..., :-1, :].contiguous()
    labels = labels[..., 1:].contiguous()
    # Just getting the size of the vocaublary (dimension of output layer)
    vocab_size = logits.size(-1)

    # number of non-ignored tokens
    num_active_elements = labels.ne(-100).sum()

    # summed CE over active tokens
    loss_sum = F.cross_entropy(
        logits.view(-1, vocab_size),
        labels.view(-1),
        ignore_index=-100,
        reduction="sum",
    )

    # Regularize over active tokens, to exactly match the built-in Huggingface implementation
    loss = loss_sum / num_active_elements
    return loss

out = demo(compute_loss_function)

Step,Training Loss
1,1.461272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
test_ds['answer'][1]

'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3'